## Central Question and Limitations
- What role do wetlands play in wildfire movement across the landscape?
- Data limitations: restricted to fires with available spatial data in the form of a vector perimeter, begining with the year 2025

# Goals for this file
- Determine how much overlap in wildfire area and wetland area exists
- Determine how many wildfires occured within 5 kilometers of a wetland
- Determine whether there is any spatial relation between our wetland data and fire data (?)
- Scale up analysis to incorporate multiple years with functions

## Data Import & Cleaning

In [2]:
# import packages
# new package so run: pip install  libpysal esda matplotlib
import platform
from pathlib import Path
import pandas as pd
import geopandas as gpd
import seaborn as sns
import calendar
import matplotlib.pyplot as plt
import pygris
import libpysal as ps # this will be used for spatial weights
import esda # this will be used for spatial autocorrelation


In [3]:
# instead of importing the cleaned fire dataset, use function to read in and clean data
def read_fires(year):
    #  SET PATH TO DATA FOLDER ON BOX DRIVE
    if platform.system() == 'Windows':
        box_path = Path.home() / 'Box'
    # if mac:
    else:
        box_path = Path.home() / "Library" / "CloudStorage" / "Box-Box"

    #Confirm that the box path is valid
    if box_path.exists():
        print(f'Box folder found at {box_path}')
    else:
        print(f'Failed to locate the Box folder.\nIs the BoxDrive app running?\nAre you logged in?')

    # create path object that goes to the specific folder on Box drive
    # uploading wetlands that are bigger than a hectare
    fire_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' / 'fire25_1.gdb'
    print(fire_path.exists())

    fire_shp = gpd.read_file(fire_path, layer='firep25_1')

    # FILTER BY DESIRED YEAR
    fire_yr = fire_shp[fire_shp['YEAR_'] == year]

    # drop rows with null values in geometry column
    fire_yr = fire_yr.dropna(subset=['geometry'])

    # add column for month of alarm date
    # add a column for month of alarm date
    # add a column for month of alarm date
    fire_yr['Alarm Month'] = fire_yr['ALARM_DATE'].dt.month

    fire_yr['Alarm Month Abb'] = pd.Categorical(
        values = fire_yr['ALARM_DATE'].dt.month_name().str[:3],
        categories=calendar.month_abbr[1:]
    )

    return fire_yr

In [4]:
# test function if year = 2025 Yay it works
fire_25 = read_fires(2025)

Box folder found at /Users/katanderson/Library/CloudStorage/Box-Box
True


/Users/katanderson/miniforge3/envs/env712/lib/python3.13/site-packages/pyogrio/raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined. Further messages of this type will be suppressed.
  return ogr_read(
/Users/katanderson/miniforge3/envs/env712/lib/python3.13/site-packages/pyogrio/raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts.  The processing may be really slow.  You can skip the processing by setting METHOD=SKIP. Further messages of this type will be suppressed.
  return ogr_read(


In [5]:
# access wetlands data

# establish paths to datasets - wetlands

# set a path to the main box folder based on machine's os
# if windows: 
if platform.system() == 'Windows':
    box_path = Path.home() / 'Box'
# if mac:
else:
    box_path = Path.home() / "Library" / "CloudStorage" / "Box-Box"

#Confirm that the box path is valid
if box_path.exists():
    print(f'Box folder found at {box_path}')
else:
    print(f'Failed to locate the Box folder.\nIs the BoxDrive app running?\nAre you logged in?')

# create path object that goes to the specific folder on Box drive
wetlands_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' / 'wetlands_subset' / 'wetlands_subset.geojson'
print(wetlands_path.exists())

# read in the wetlands geojson file, this will take a WHILE - 10 mins 1st try 
wetlands = gpd.read_file(wetlands_path, engine="pyogrio")

Box folder found at /Users/katanderson/Library/CloudStorage/Box-Box
True


In [6]:
# write function to put both datasets in the same CRS
def convert_crs(shp1, shp2):
    # CHECK CRS OF BOTH
    print(shp1.crs)
    print(shp2.crs)

    # PUT IN EPSG:26910 FOR UTM ZONE 10N IN CA
    shp1 = shp1.to_crs(epsg=26910)
    shp2 = shp2.to_crs(epsg=26910)

    print(shp1.crs)
    print(shp2.crs)

    return shp1, shp2


In [ ]:
# test CRS function
fire_25, wetlands = convert_crs(fire_25, wetlands)


EPSG:3310
EPSG:5070
EPSG:26910
EPSG:26910


## Generate Functions for Analysis by Year

### Distance to Nearest Fire

In [ ]:
# DISTANCE TO NEAREST FIRE
def dist_fire(wetlands_shp, fire_shp):
    # Nearest Neighbor
    nearest_fires = gpd.sjoin_nearest(
        wetlands_shp, fire_shp, how='left', distance_col='Dist_Closest_m'
    )

    # convert distance column from meters to kilometers
    nearest_fires['Dist_Closest_km'] = nearest_fires['Dist_Closest_m'] / 1000

    # add information to a table called fire_stats
    fire_stats = (nearest_fires.groupby('C System')['Dist_Closest_km'].agg(
        Avg_Dist_Fire_km="mean",
        Std_Dist_Fire_km="std",
        Median_Dist_Fire_km="median",
        Range_Dist_Fire_km=lambda x: x.max() - x.min(),
        IQR_Dist_Fire_km=lambda x: x.quantile(0.75) - x.quantile(0.25),
        ).reset_index()
    )

    return nearest_fires, fire_stats
# outputs are a shapefile and the fire_stats table

In [10]:
# test if distance function works
nearest_fires, fire_stats = dist_fire(wetlands, fire_25)

### Overlap Between Wetlands and Fires

In [11]:
# JOIN FIRE AND WETLAND DATA
# inputs for this function are both shapefiles and the statistics table we are adding to
def overlap(wetlands_shp, fire_shp, fire_stats):
    
    # calculate the total area of fire in CA in hectares
    fire_area = fire_shp.geometry.area.sum() / 10000
    
    # calculate the total area of wetlands in CA
    wetlands_area = wetlands_shp.geometry.area.sum() / 10000

    print(f'AREA OF ALL FIRE: {fire_area} hectares')
    print(f'AREA OF ALL WETLANDS: {wetlands_area} hectares')
    
    # PERFORM SPATIAL JOIN
    intersect = gpd.overlay(wetlands_shp, fire_shp, how="intersection")

    # add a column to the intersected SHP that shows intersect area in hectares
    # 1 hectare = 10,000 square meters
    intersect['int_area_ha']=intersect.geometry.area / 10000

    # ADD WETLAND AREA BURNED BY SYSTEM TO FIRE_STATS TABLE
    burned = intersect.groupby('C System')['int_area_ha'].sum()
    fire_stats['Burned Area (ha)'] = fire_stats['C System'].map(burned)

    # ADD PCT WETLAND BURNED BY SYSTEM
    # get wetland area by system
    total_wetlands = wetlands_shp.groupby('C System')['geometry'].apply(lambda x: x.area.sum() / 10000)

    # sanity check - total wetland area does match the sum of each system
    print(round(total_wetlands.sum()) == round(wetlands_area))

    # add wetland area to fire stats table
    fire_stats['Total Wetland Area (ha)'] = fire_stats['C System'].map(total_wetlands)

    fire_stats['Percent Wetland Area Burned'] = (fire_stats['Burned Area (ha)'] / fire_stats['Total Wetland Area (ha)']) *100

    return intersect, fire_stats

In [12]:
# test the overlap function
intersect, fire_stats = overlap(wetlands, fire_25, fire_stats)

AREA OF ALL FIRE: 206320.6875409995 hectares
AREA OF ALL WETLANDS: 3382322.7053495385 hectares
True


In [13]:
fire_stats

,C System,Avg_Dist_Fire_km,Std_Dist_Fire_km,Median_Dist_Fire_km,Range_Dist_Fire_km,IQR_Dist_Fire_km,Burned Area (ha),Total Wetland Area (ha),Percent Wetland Area Burned
0,Estuarine,22.976303,20.434598,17.234116,77.377007,20.387559,0.175689,1.638092e+05,0.000107
1,Lacustrine,21.800034,16.374344,18.945761,184.902020,18.550049,137.867565,7.862960e+05,0.017534
2,Marine,28.184545,18.768883,26.063212,111.962243,21.480902,4.505302,1.342564e+06,0.000336
3,Palustrine,23.656434,23.122075,17.964862,189.129597,19.731767,1349.317309,6.795560e+05,0.198559
4,Riverine,23.102199,25.251692,15.678355,191.569958,20.699499,1330.207814,4.100976e+05,0.324364
